<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://sebastianraschka.com">Sebastian Raschka</a> 所著《<a href="https://mng.bz/lZ5B">构建推理模型（从零开始）</a>》一书的补充代码<br>
<br>代码仓库：<a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 附录 F：LLM 评估的常见方法

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.0
torch version: 2.7.1
tokenizers version: 0.21.2


&nbsp;
## F.1 理解大语言模型的主要评估方法

- 本节无代码

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F01_raschka.webp" width="500px">

&nbsp;
### F.2 评估答案选项准确性

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F02_raschka.webp" width="500px" alt="附录 F 图 F-2">

- 请注意，此图展示的是基于多选题评估（如MMLU）的简化版本，其中我们将生成的输出字母与正确答案字母进行比对
- 实际应用中，其变体包括对数概率评分法，即不仅检查最终字母，还会计算模型对每个候选答案的置信度
- 对于推理模型，这还可能涉及评估当正确答案输入模型时被生成的可能性
- 无论采用何种方式，评估仍会检查模型是否选择了预定义答案之一
- （输出概率评分将在第4章详细讨论，届时我们将改进文本生成函数）

&nbsp;
#### F.2.1 加载模型

In [2]:
from pathlib import Path
import torch

from reasoning_from_scratch.ch02 import (
    get_device
)
from reasoning_from_scratch.qwen3 import (
    download_qwen3_small,
    Qwen3Tokenizer,
    Qwen3Model,
    QWEN_CONFIG_06_B
)

device = get_device()
torch.set_float32_matmul_precision("high")

# If you have compatibility issues, try to
# uncomment the line below and rerun the notebook
# device = "cpu"

WHICH_MODEL = "base"

if WHICH_MODEL == "base":

    download_qwen3_small(
        kind="base", tokenizer_only=False, out_dir="qwen3"
    )

    tokenizer_path = Path("qwen3") / "tokenizer-base.json"
    model_path = Path("qwen3") / "qwen3-0.6B-base.pth"
    tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

elif WHICH_MODEL == "reasoning":

    download_qwen3_small(
        kind="reasoning", tokenizer_only=False, out_dir="qwen3"
    )

    tokenizer_path = Path("qwen3") / "tokenizer-reasoning.json"
    model_path = Path("qwen3") / "qwen3-0.6B-reasoning.pth"
    tokenizer = Qwen3Tokenizer(
        tokenizer_file_path=tokenizer_path,
        apply_chat_template=True,
        add_generation_prompt=True,
        add_thinking=True,
    )

else:
    raise ValueError(f"Invalid choice: WHICH_MODEL={WHICH_MODEL}")


model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(model_path))

model.to(device)


USE_COMPILE = False  # Set to true to enable compilation
if USE_COMPILE:
  torch._dynamo.config.allow_unspec_int_on_nn_module = True
  model = torch.compile(model)

Using Apple Silicon GPU (MPS)
✓ qwen3/qwen3-0.6B-base.pth already up-to-date
✓ qwen3/tokenizer-base.json already up-to-date


&nbsp;
#### F.2.2 检查生成的答案信函

In [3]:
example = {
    "question": (
        "How many ways are there to put 4 distinguishable"
        " balls into 2 indistinguishable boxes?"
    ),
    "choices": ["7", "11", "16", "8"],
    "answer": "D",
}

def format_prompt(example):
    return (
        f"{example['question']}\n"
        f"A. {example['choices'][0]}\n"
        f"B. {example['choices'][1]}\n"
        f"C. {example['choices'][2]}\n"
        f"D. {example['choices'][3]}\n"
        "Answer: "  # trailing space encourages a single-letter next token
    )

prompt = format_prompt(example)
print(prompt)

How many ways are there to put 4 distinguishable balls into 2 indistinguishable boxes?
A. 7
B. 11
C. 16
D. 8
Answer: 


---

- 您可以通过 `datasets` 库直接加载 MMLU 数据集中的示例（可通过 `pip install datasets` 或 `uv add datasets` 安装）：

```python
from datasets import load_dataset

configs = get_dataset_config_names("cais/mmlu")
dataset = load_dataset("cais/mmlu", "high_school_mathematics")

# 查看测试集中的第一个示例：
example = dataset["test"][0]
print(example)
```

- 上述代码使用了 `"high_school_mathematics"` 子集；要获取其他子集的列表，请使用以下代码：

```python
from datasets import get_dataset_config_names

subsets = get_dataset_config_names("cais/mmlu")
print(subsets)
```

---

In [4]:
prompt_ids = tokenizer.encode(prompt)
prompt_fmt = torch.tensor(prompt_ids, device=device).unsqueeze(0)

- 我们生成几个 token，并提取模型输出中首次出现的字母 A/B/C/D：

In [5]:
from reasoning_from_scratch.ch02 import generate_text_basic_stream_cache


def predict_choice(
    model, tokenizer, prompt_fmt, max_new_tokens=8
):
    pred = None
    for t in generate_text_basic_stream_cache(
        model=model,
        token_ids=prompt_fmt,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id,
    ):
        answer = tokenizer.decode(t.squeeze(0).tolist())
        for letter in answer:
            letter = letter.upper()
            if letter in "ABCD":
                pred = letter
                break
        if pred:  # stop as soon as a letter appears
            break
    return pred

In [6]:
pred1 = predict_choice(model, tokenizer, prompt_fmt)

print(
    f"Generated letter: {pred1}\n"
    f"Correct? {pred1 == example['answer']}"
)

Generated letter: C
Correct? False


&nbsp;
### F.3 使用验证器检查答案

- 本节无代码（见第3章）

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F03_raschka.webp" width="500px">

<br>
&nbsp;

### F.4 使用偏好和排行榜比较模型

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F04_raschka.webp" width="500px">

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F05_raschka.webp" width="500px" alt="附录F 图5">

- Elo 评分（"400 分算法"）灵感源自国际象棋排名：https://en.wikipedia.org/wiki/Performance_rating_(chess)
- 需注意，LM Arena 已转向采用统计学 Bradley-Terry 模型，该模型提供类似 Elo 量表的分数；然而，成对排名的核心概念仍然适用

In [7]:
# Pairwise "arena votes" where the first model is the winner and
# the second model is the loser
votes = [
    ("GPT-5", "Claude-3"),  # First match-up: GPT-5 was preferred over Claude-3
    ("GPT-5", "Llama-4"),
    ("Claude-3", "Llama-3"),
    ("Llama-4", "Llama-3"),
    ("Claude-3", "Llama-3"),
    ("GPT-5", "Llama-3"),
]

In [8]:
def elo_ratings(vote_pairs, k_factor=32, initial_rating=1000):
    # Initialize all models with the same base rating
    ratings = {
        model: initial_rating
        for pair in vote_pairs
        for model in pair
    }

    # Update ratings after each match
    for winner, loser in vote_pairs:

        # Expected score for the current winner given the ratings
        expected_winner = 1.0 / (
            1.0 + 10 ** ((ratings[loser] - ratings[winner]) / 400.0)
        )

        # k_factor determines sensitivity of rating updates
        ratings[winner] = (
            ratings[winner] + k_factor * (1 - expected_winner)
        )
        ratings[loser] = (
            ratings[loser] + k_factor * (0 - (1 - expected_winner))
        )

    return ratings

In [9]:
ratings = elo_ratings(votes, k_factor=32, initial_rating=1000)

for model in sorted(ratings, key=ratings.get, reverse=True):
    print(f"{model:8s} : {ratings[model]:.1f}")

GPT-5    : 1043.7
Claude-3 : 1015.2
Llama-4  : 1000.7
Llama-3  : 940.4


- 预期获胜方的得分计算如下：

$$\text{expected\_winner} \;=\; \frac{1}{1 + 10^{\tfrac{\text{rating\_loser} - \text{rating\_winner}}{400}}}
$$

- 直观理解：
    - 若 rating_winner >> rating_loser：
       - 指数 → 非常负
       - 分母 ≈ 1
       - expected_winner ≈ 1（几乎确定获胜）
    - 若 rating_winner << rating_loser：
       - 指数 → 非常正
       - 分母 → 非常大
       - expected_winner ≈ 0（几乎确定失败）
    - 若 rating_winner == rating_loser：
       - 指数 = 0
       - 分母 = 2
       - expected_winner = 0.5（势均力敌）

&nbsp;
### F.5 使用其他LLM评估回复

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F06_raschka.webp" width="500px">

- 在本节中，我们使用另一个更大的LLM来自动化微调后LLM的响应评估
- 具体而言，我们使用了Open AI开发的指令微调200亿参数gpt-oss模型，该模型可通过ollama在本地运行（[https://ollama.com](https://ollama.com)）

- Ollama 是一个用于高效运行大语言模型的开源应用程序
- 它是 llama.cpp（[https://github.com/ggerganov/llama.cpp](https://github.com/ggerganov/llama.cpp)）的封装工具，该库通过纯 C/C++ 实现大语言模型以最大化效率
- 需要注意的是，这是一个用于使用大语言模型生成文本（推理）的工具，而非用于训练或微调大语言模型
- 在运行下方代码前，请访问 [https://ollama.com](https://ollama.com) 并按照说明安装 Ollama（例如，点击“Download”按钮并下载适用于您操作系统的 Ollama 应用程序）

- 对于 macOS 和 Windows 用户，点击您下载的 ollama 应用程序；如果提示您安装命令行使用方式，请选择“是”
- Linux 用户可以使用 ollama 网站上提供的安装命令
- 我们可以在电脑上通过 3 种方式运行 ollama：

**1. `ollama serve`**

- 此命令将 ollama 后端作为服务器运行，通常监听 `http://localhost:11434`。它不会立即加载模型，直到我们通过 API 调用。如果我们想通过 Python 使用 ollama，这正是我们需要的方式。

**2. `ollama run gpt-oss:20b`**

- 这是一个便捷的封装命令。如果服务器尚未运行，它会先启动服务器，然后下载模型（首次运行时），并让我们进入一个交互式终端，在那里可以与模型聊天。在后台，它使用的是相同的服务器 API。

**3. Ollama 桌面应用**

- 它会自动运行相同的后端，并在其上提供一个图形用户界面（如上图所示）。
它还会应用默认设置（系统提示词、温度参数、停止序列），这可以解释为什么其回答看起来与直接使用原始 API 的结果有所不同。

---

**注意**：

- 在终端中运行 `ollama serve` 时（如上所述），可能会遇到错误信息：`Error: listen tcp 127.0.0.1:11434: bind: address already in use`
- 如果出现这种情况，请尝试使用命令 `OLLAMA_HOST=127.0.0.1:11435 ollama serve`（如果该地址也被占用，请尝试将数字递增，直到找到未被占用的地址）

---

- 例如，要尝试使用 ollama，我们可以运行 `ollama run gpt-oss:20b` 来体验拥有 200 亿参数的 gpt-oss 20B 模型。该模型（约 13 GB）将在您首次运行此命令时自动下载。（或者，您也可以像上图所示那样在桌面应用中使用它。）

```bash
ollama run gpt-oss:20b
```

- 输出结果如下所示：

```
$ ollama run gpt-oss:20b
pulling manifest 
pulling b112e727c6f1: 100% ▕█████████████████████████████████▏  13 GB                         
pulling fa6710a93d78: 100% ▕█████████████████████████████████▏ 7.2 KB                         
pulling f60356777647: 100% ▕█████████████████████████████████▏  11 KB                         
pulling d8ba2f9a17b3: 100% ▕█████████████████████████████████▏   18 B                         
pulling 55c108d8e936: 100% ▕█████████████████████████████████▏  489 B                         
verifying sha256 digest 
writing manifest 
removing unused layers 
success
```

- 关于 gpt-oss 的更多信息，请参阅我的深度分析文章：[从 GPT-2 到 gpt-oss：架构演进分析](https://magazine.sebastianraschka.com/p/from-gpt-2-to-gpt-oss-analyzing-the)
- 使用 ollama 运行 `"gpt-oss:20b"` 模型（一个 200 亿参数的模型）需要 13 GB 内存；如果您的机器不支持，可以尝试更小的模型，例如 40 亿参数的 `qwen3:4b` 模型，它仅需约 4 GB 内存
- 或者，如果您的机器支持，您也可以使用更大的 1200 亿参数的 gpt-oss（`qwen3:235b`）甚至 2350 亿参数的 Qwen3 模型（`qwen3:235b`）
- 下载完成后，您将看到一个命令行提示符，可以与模型进行对话
- 尝试输入类似 "What is 1+2?" 的提示，它应该会返回如下所示的输出

```
>>> What is 1+2?
Thinking...
User asks: "What is 1+2?" This is simple: answer 3. Provide explanation? Possibly ask for simple 
arithmetic. Provide answer: 3.
...done thinking.

1 + 2 = **3**
```

- 您可以通过输入 `/bye` 来结束当前会话

- 以下代码会在使用 ollama 评估我们之前生成的测试集响应之前，检查 ollama 会话是否正常运行

In [10]:
import psutil

def check_if_running(process_name):
    running = False
    for proc in psutil.process_iter(["name"]):
        if process_name in proc.info["name"]:
            running = True
            break
    return running

ollama_running = check_if_running("ollama")

if not ollama_running:
    raise RuntimeError(
        "Ollama not running. Launch ollama before proceeding."
    )
print("Ollama running:", check_if_running("ollama"))

Ollama running: True


- 现在，除了我们之前使用的 `ollama run` 命令与模型交互外，还可以通过 Python 中的 REST API 使用以下函数来实现
- 在运行本笔记本的下一个代码单元之前，请确保 ollama 仍在运行（之前的代码单元应输出 `"Ollama running: True"`）
- 接下来，运行以下代码单元以查询模型

In [11]:
import json
import requests


def query_model(
    prompt,
    model="gpt-oss:20b",
    # If you used OLLAMA_HOST=127.0.0.1:11435 ollama serve
    # update the address from 11434 to 11435
    url="http://localhost:11434/api/chat"
):
    # Create the data payload as a dictionary
    data = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "options": {     # Settings below are required for deterministic responses
            "seed": 123,
            "temperature": 0,
            "num_ctx": 2048
        }
    }

    # Send the POST request
    with requests.post(url, json=data, stream=True, timeout=30) as r:
        r.raise_for_status()
        response_data = ""
        for line in r.iter_lines(decode_unicode=True):
            if not line:
                continue
            response_json = json.loads(line)
            if "message" in response_json:
                response_data += response_json["message"]["content"]

    return response_data

In [12]:
ollama_model = "gpt-oss:20b"
result = query_model("What is 1+2?", ollama_model)
print(result)

3


- 现在，使用我们上面定义的 `query_model` 函数，我们可以评估我们自己模型的响应。

In [16]:
def rubric_prompt(instruction, reference_answer, model_answer):
    rubric = (
        "You are a fair judge assistant. You will be given an instruction, "
        "a reference answer, and a candidate answer to evaluate, according "
        "to the following rubric:\n\n"
        "1: The response fails to address the instruction, providing "
        "irrelevant, incorrect, or excessively verbose content.\n"
        "2: The response partially addresses the instruction but contains "
        "major errors, omissions, or irrelevant details.\n"
        "3: The response addresses the instruction to some degree but is "
        "incomplete, partially correct, or unclear in places.\n"
        "4: The response mostly adheres to the instruction, with only "
        "minor errors, omissions, or lack of clarity.\n"
        "5: The response fully adheres to the instruction, providing a "
        "clear, accurate, and relevant answer in a concise and efficient "
        "manner.\n\n"
        "Now here is the instruction, the reference answer, and the "
        "response.\n"
    )

    prompt = (
        f"{rubric}\n"
        f"Instruction:\n{instruction}\n\n"
        f"Reference Answer:\n{reference_answer}\n\n"
        f"Answer:\n{model_answer}\n\n"
        f"Evaluation: "
    )
    return prompt

- `model_answer` 可以是我们自己模型生成的答案；为简化起见，这里我们硬编码一个可能的模型答案

In [17]:
rendered_prompt = rubric_prompt(
    instruction=(
        "If all birds can fly, and a penguin is a bird, "
        "can a penguin fly?"
    ),
    reference_answer=(
        "Yes, according to the premise that all birds can fly, "
        "a penguin can fly."
    ),
    model_answer=(
        "Yes – under those premises a penguin would be able to fly."
    )
)
print(rendered_prompt)

You are a fair judge assistant. You will be given an instruction, a reference answer, and a candidate answer to evaluate, according to the following rubric:

1: The response fails to address the instruction, providing irrelevant, incorrect, or excessively verbose content.
2: The response partially addresses the instruction but contains major errors, omissions, or irrelevant details.
3: The response addresses the instruction to some degree but is incomplete, partially correct, or unclear in places.
4: The response mostly adheres to the instruction, with only minor errors, omissions, or lack of clarity.
5: The response fully adheres to the instruction, providing a clear, accurate, and relevant answer in a concise and efficient manner.

Now here is the instruction, the reference answer, and the response.

Instruction:
If all birds can fly, and a penguin is a bird, can a penguin fly?

Reference Answer:
Yes, according to the premise that all birds can fly, a penguin can fly.

Answer:
Yes – 

In [18]:
result = query_model(rendered_prompt, ollama_model)
print(result)

**Score: 5**

The candidate answer directly addresses the question, correctly applies the given premises, and concisely states that a penguin would be able to fly. It is accurate, relevant, and clear.
